In [2]:
import scanpy as sc

/home/krupavardhan4869@gmail.com/disenv/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [4]:
adata = sc.read_h5ad('../combined_data.h5ad')

In [5]:
adata

AnnData object with n_obs × n_vars = 1560000 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types', '_scvi_batch', '_scvi_labels'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'log1p', 'pca'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scanorama', 'X_scetm', 'X_scvi', 'theta'
    varm: 'PCs'

In [6]:
# Setup ccAF for cell cycle prediction
import math
import numpy as np
import pandas as pd
import types
import tensorflow as tf

# TensorFlow 1.x compatibility for ccAF
tf.compat.v1.disable_eager_execution()
tf.placeholder = tf.compat.v1.placeholder
tf.Session = tf.compat.v1.Session

import ccAF

# Fix ccAF bug (set used as index in prep)
def _fixed_prep_predict_data(self, data):
    missing = set(self.genes).difference(data.index)
    if len(missing) > 0:
        data = pd.concat([data, pd.DataFrame(0, index=list(missing), columns=data.columns)])
    return data.loc[list(self.genes)]

ccAF.ccAF._Classifier_ACTINN__prep_predict_data = types.MethodType(
    _fixed_prep_predict_data, ccAF.ccAF
)

print("ccAF loaded and fixed")


2025-11-14 10:48:48.469807: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ccAF loaded and fixed


In [9]:
# Helper function to build ccAF AnnData from embedding
def build_ccaf_ann_from_embedding(embedding_matrix: np.ndarray, obs_index) :
    """Create a ccAF-shaped AnnData using an embedding matrix as X."""
    import anndata as ad
    ccaf_genes = list(ccAF.ccAF.genes)
    k = len(ccaf_genes)  # usually 1472
    n, d = embedding_matrix.shape
    if d >= k:
        Xk = embedding_matrix[:, :k]
    else:
        pad = np.zeros((n, k - d), dtype=embedding_matrix.dtype)
        Xk = np.hstack([embedding_matrix, pad])
    var_df = pd.DataFrame(index=ccaf_genes)
    ad_emb = ad.AnnData(X=Xk, obs=pd.DataFrame(index=obs_index), var=var_df)
    return ad_emb

# Helper function for batched predictions
def ccaf_predict_batched(adata_like, batch_size: int = 50000) -> np.ndarray:
    """Run ccAF predictions in batches."""
    n = adata_like.n_obs
    out = []
    steps = math.ceil(n / batch_size)
    print(f"Running ccAF predictions: {n} cells in {steps} batches")
    for i in range(steps):
        s = i * batch_size
        e = min((i + 1) * batch_size, n)
        print(f"  Batch {i+1}/{steps}: rows {s}:{e}")
        sub = adata_like[s:e, :].copy()
        out.extend(ccAF.ccAF.predict_labels(sub))
    return np.array(out)


In [10]:
# Get cell cycle predictions from X_pca
print("Getting cell cycle predictions from X_pca...")
X_pca = adata.obsm['X_pca']
ad_pca = build_ccaf_ann_from_embedding(X_pca, obs_index=adata.obs_names)
cell_cycle_predictions = ccaf_predict_batched(ad_pca, batch_size=50000)

# Add as obs column
adata.obs['cell_cycle'] = cell_cycle_predictions
print(f"\nCell cycle predictions added to adata.obs['cell_cycle']")
print(f"Unique cell cycle types: {adata.obs['cell_cycle'].unique()}")
print(f"\nCell cycle distribution:")
print(adata.obs['cell_cycle'].value_counts())


Getting cell cycle predictions from X_pca...
Running ccAF predictions: 1560000 cells in 32 batches
  Batch 1/32: rows 0:50000


I0000 00:00:1763117368.186643   19707 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14940 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:06:00.0, compute capability: 7.5
I0000 00:00:1763117368.193478   19707 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


  Batch 2/32: rows 50000:100000


I0000 00:00:1763117371.505636   19707 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14940 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:06:00.0, compute capability: 7.5


  Batch 3/32: rows 100000:150000


...

  Batch 31/32: rows 1500000:1550000


I0000 00:00:1763117446.348961   19707 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14940 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:06:00.0, compute capability: 7.5


  Batch 32/32: rows 1550000:1560000


I0000 00:00:1763117447.523608   19707 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14940 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:06:00.0, compute capability: 7.5



Cell cycle predictions added to adata.obs['cell_cycle']
Unique cell cycle types: ['S/G2' 'G1' 'M/Early G1' 'Late G1' 'G1/other' 'S' 'G2/M' 'Neural G0']

Cell cycle distribution:
cell_cycle
G1            647682
Late G1       424470
G2/M          159340
S             108090
G1/other       78668
S/G2           77723
Neural G0      55125
M/Early G1      8902
Name: count, dtype: int64


In [ ]:
# Compute UMAPs for each embedding
import matplotlib.pyplot as plt
import anndata as ad

embeddings_to_plot = ['X_pca_harmony', 'X_scanorama', 'X_scvi', 'X_scetm']

# Check which embeddings are available
available_embeddings = [emb for emb in embeddings_to_plot if emb in adata.obsm.keys()]
print(f"Available embeddings: {available_embeddings}")

# Compute UMAP for each embedding
for emb_name in available_embeddings:
    print(f"\nComputing UMAP for {emb_name}...")
    # Create temporary AnnData with this embedding
    adata_temp = ad.AnnData(obs=adata.obs.copy())
    adata_temp.obsm[emb_name] = adata.obsm[emb_name]
    
    # Compute UMAP (scanpy will use the embedding in obsm)
    sc.pp.neighbors(adata_temp, use_rep=emb_name, n_neighbors=15, n_pcs=None)
    sc.tl.umap(adata_temp)
    
    # Store UMAP coordinates
    adata.obsm[f'X_umap_{emb_name}'] = adata_temp.obsm['X_umap']
    print(f"  UMAP computed and stored in obsm['X_umap_{emb_name}']")


Available embeddings: ['X_pca_harmony']

Computing UMAP for X_pca_harmony...


In [ ]:
# Plot UMAPs colored by cell cycle type
n_embeddings = len(available_embeddings)
n_cols = 2
n_rows = (n_embeddings + 1) // 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 6 * n_rows))
if n_embeddings == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for idx, emb_name in enumerate(available_embeddings):
    ax = axes[idx]
    
    # Get UMAP coordinates
    umap_key = f'X_umap_{emb_name}'
    umap_coords = adata.obsm[umap_key]
    
    # Create scatter plot colored by cell cycle
    scatter = ax.scatter(umap_coords[:, 0], umap_coords[:, 1], 
                        c=pd.Categorical(adata.obs['cell_cycle']).codes,
                        s=0.5, alpha=0.6, cmap='tab20')
    
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(f'UMAP: {emb_name}\n(colored by cell cycle)')
    ax.set_aspect('equal')
    
    # Add colorbar with cell cycle labels
    unique_cycles = adata.obs['cell_cycle'].unique()
    cbar = plt.colorbar(scatter, ax=ax, ticks=range(len(unique_cycles)))
    cbar.set_ticklabels(unique_cycles)
    cbar.set_label('Cell Cycle Type', rotation=270, labelpad=15)

# Hide extra subplots if any
for idx in range(n_embeddings, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('umap_cell_cycle_by_embedding.png', dpi=200, bbox_inches='tight')
plt.show()
print("\nUMAP plots saved to 'umap_cell_cycle_by_embedding.png'")


In [ ]:
# Alternative: Plot each UMAP separately with better color scheme
import seaborn as sns

for emb_name in available_embeddings:
    umap_key = f'X_umap_{emb_name}'
    umap_coords = adata.obsm[umap_key]
    
    plt.figure(figsize=(10, 8))
    
    # Create a dataframe for easier plotting
    plot_df = pd.DataFrame({
        'UMAP_1': umap_coords[:, 0],
        'UMAP_2': umap_coords[:, 1],
        'cell_cycle': adata.obs['cell_cycle']
    })
    
    # Plot with seaborn for better color handling
    unique_cycles = sorted(plot_df['cell_cycle'].unique())
    palette = sns.color_palette("tab20", n_colors=len(unique_cycles))
    
    for i, cycle_type in enumerate(unique_cycles):
        subset = plot_df[plot_df['cell_cycle'] == cycle_type]
        plt.scatter(subset['UMAP_1'], subset['UMAP_2'], 
                   label=cycle_type, s=0.5, alpha=0.6, color=palette[i])
    
    plt.xlabel('UMAP 1', fontsize=12)
    plt.ylabel('UMAP 2', fontsize=12)
    plt.title(f'UMAP: {emb_name} (colored by cell cycle type)', fontsize=14, fontweight='bold')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'umap_{emb_name}_cell_cycle.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: umap_{emb_name}_cell_cycle.png")
